In [1]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install fastparquet

   ---------------------------------------- 0.0/667.4 kB ? eta -:--:--
   --------------------------------------- 667.4/667.4 kB 15.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 9.3 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]

Note: you may need to restart the kernel to use updated packages.


# LEVEL MACRO

In [4]:
import pandas as pd

df0_bus = pd.read_parquet("Business_counts_IS8_LADs.parquet")
df0_em = pd.read_parquet("Employee_counts_IS8_LADs.parquet")

keys = ["YEAR","GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "IS8_SECTOR", "FRONTIER_SECTOR"]

df0_bus = df0_bus.dropna(subset=["IS8_SECTOR"])
df0_em = df0_em.dropna(subset=["IS8_SECTOR"])

# I needed to groupby per Sector because the database was too heavy. 

df0_bus_grouped = df0_bus.groupby(keys, as_index=False)["OBS_VALUE"].sum()
df0_bus_grouped = df0_bus_grouped.rename(columns={"OBS_VALUE": "OBS_Value_Business"})

df0_em_grouped = df0_em.groupby(keys, as_index=False)["OBS_VALUE"].sum()
df0_em_grouped = df0_em_grouped.rename(columns={"OBS_VALUE": "OBS_Value_Employment"})

df_merged = df0_bus_grouped.merge(
    df0_em_grouped,
    on=keys,
    how="left"
)

for col in keys:
    df_merged[col] = df_merged[col].astype("category")

df_merged["OBS_Value_Business"] = df_merged["OBS_Value_Business"].astype("float32")
df_merged["OBS_Value_Employment"] = df_merged["OBS_Value_Employment"].astype("float32")

FileNotFoundError: [Errno 2] No such file or directory: 'Business_counts_IS8_LADs.parquet'

# LEVEL MICRO

In [57]:
import pandas as pd

df1_bus = pd.read_parquet("Business_counts_IS8_MSOAs.parquet")
df1_em = pd.read_parquet("Employee_counts_IS8_MSOAs.parquet")


keys = ["YEAR","GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "IS8_SECTOR", "FRONTIER_SECTOR"]

df1_bus = df1_bus.dropna(subset=["IS8_SECTOR"])
df1_em = df1_em.dropna(subset=["IS8_SECTOR"])

# I needed to groupby per Sector because the database was too heavy. 

df1_bus_grouped = df1_bus.groupby(keys, as_index=False)["OBS_VALUE"].sum()
df1_bus_grouped = df1_bus_grouped.rename(columns={"OBS_VALUE": "OBS_Value_Business"})

df1_em_grouped = df1_em.groupby(keys, as_index=False)["OBS_VALUE"].sum()
df1_em_grouped = df1_em_grouped.rename(columns={"OBS_VALUE": "OBS_Value_Employment"})

df_merged_msoa = df1_bus_grouped.merge(
    df1_em_grouped,
    on=keys,
    how="left"
)

for col in keys:
    df_merged_msoa[col] = df_merged_msoa[col].astype("category")

df_merged_msoa["OBS_Value_Business"] = df_merged_msoa["OBS_Value_Business"].astype("float32")
df_merged_msoa["OBS_Value_Employment"] = df_merged_msoa["OBS_Value_Employment"].astype("float32")

df_merged_msoa.head()

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment
0,2016,E02000984,Bolton 001,Advanced manufacturing,Aerospace manufacturing,0.0,0.0
1,2016,E02000984,Bolton 001,Advanced manufacturing,Agritech,0.0,0.0
2,2016,E02000984,Bolton 001,Advanced manufacturing,Automotive manufacturing,0.0,0.0
3,2016,E02000984,Bolton 001,Advanced manufacturing,Batteries,0.0,10.0
4,2016,E02000984,Bolton 001,Creative Industries,Advertising and marketing,5.0,5.0


# MERGE WITH DEMOGRAPHIC DATA

In [27]:
import pandas as pd

file = "ONS_local_indicators_package.xlsx"

# Skip first 4 sheets
xls = pd.ExcelFile(file)
sheet_names = xls.sheet_names[4:]

dfs = []

for sheet_name in sheet_names:
    # Read the whole sheet first as raw
    raw_df = pd.read_excel(file, sheet_name=sheet_name, header=None)
    
    # Find the row where 'Area Code' appears
    area_row_idx = raw_df[raw_df.iloc[:,0].astype(str).str.contains("Area Code", na=False)].index
    if len(area_row_idx) == 0:
        print(f"Skipping sheet '{sheet_name}' because no 'Area Code' found")
        continue
    header_row = area_row_idx[0]
    
    # Now read the sheet again using the correct header
    df = pd.read_excel(file, sheet_name=sheet_name, header=header_row)
    
    # Drop completely empty columns
    df = df.dropna(axis=1, how='all')
    
    # Identify numeric columns (including numbers stored as text)
    numeric_cols = []
    for col in df.columns:
        temp = pd.to_numeric(df[col], errors='coerce')
        if temp.notna().any():
            numeric_cols.append(col)
    
    if not numeric_cols:
        continue  # skip if no numeric column
    
    # Identifier columns
    id_cols = [c for c in df.columns if c not in numeric_cols]
    
    # Melt numeric columns into long format
    df_long = df.melt(
        id_vars=id_cols,
        value_vars=numeric_cols,
        var_name="Metric",
        value_name="Metric_Value"
    )
    
    # Ensure Metric_Value is numeric
    df_long["Metric_Value"] = pd.to_numeric(df_long["Metric_Value"], errors='coerce')
    
    # Add sheet name for reference
    df_long["Sheet"] = sheet_name
    
    dfs.append(df_long)

# Concatenate all sheets
df_all = pd.concat(dfs, ignore_index=True)

# Convert codes to category to save memory
code_cols = [c for c in df_all.columns if "Code" in c or "Area" in c]
for col in code_cols:
    df_all[col] = df_all[col].astype("category")

print(df_all.info())
df_all.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26240 entries, 0 to 26239
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   Area Code                          26240 non-null  category
 1   ITL Level 1                        54 non-null     object  
 2   County or Unitary Authority        6086 non-null   object  
 3   Local Authority District           18448 non-null  object  
 4   Notes                              963 non-null    object  
 5   Metric                             26240 non-null  object  
 6   Metric_Value                       25852 non-null  float64 
 7   Sheet                              26240 non-null  object  
 8   Country                            74 non-null     object  
 9   Nation                             196 non-null    object  
 10  Region                             650 non-null    object  
 11  Combined Authority or City Region  348 no

,Area Code,ITL Level 1,County or Unitary Authority,Local Authority District,Notes,Metric,Metric_Value,Sheet,Country,Nation,Region,Combined Authority or City Region,ITL Level 2,ITL Level 3,Observation Status,Welsh Health Board,Data accuracy,Police Force Area
0,E06000047,NaN,County Durham,NaN,NaN,Gross Value Added (GVA) per hour worked (£),30.64,GVA per hour,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E06000005,NaN,Darlington,NaN,NaN,Gross Value Added (GVA) per hour worked (£),28.56,GVA per hour,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E06000001,NaN,Hartlepool,NaN,NaN,Gross Value Added (GVA) per hour worked (£),29.84,GVA per hour,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E06000002,NaN,Middlesbrough,NaN,NaN,Gross Value Added (GVA) per hour worked (£),29.50,GVA per hour,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E06000057,NaN,Northumberland,NaN,NaN,Gross Value Added (GVA) per hour worked (£),30.43,GVA per hour,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:

# Ensure both are strings
df_all["Area Code"] = df_all["Area Code"].astype(str).str.strip()
df_merged["GEOGRAPHY_CODE"] = df_merged["GEOGRAPHY_CODE"].astype(str).str.strip()
# Codes in df_all that exist in MSOA
matched = df_all["Area Code"].isin(df_merged["GEOGRAPHY_CODE"])
print(f"Matched: {matched.sum()} / {len(df_all)} rows")

Matched: 21753 / 26240 rows


In [ ]:
df_grouped = (
    df_all
    .groupby(["Area Code", "Metric"], as_index=False)
    .agg({
        "Metric_Value": "mean" 
    })
)

df_grouped.head()

,Area Code,Metric,Metric_Value
0,E06000001,19+ further education and skills participation...,9800.000000
1,E06000001,Age-standardised mortality rate for those aged...,37.852970
2,E06000001,Apprenticeships achieved by adults aged 16+ ba...,601.000000
3,E06000001,Apprenticeships started by adults aged 16+ bas...,1173.000000
4,E06000001,Average travel time in minutes to reach neares...,7.790802


In [59]:
import pandas as pd

# Ensure codes are strings and stripped
df_grouped["Area Code"] = df_grouped["Area Code"].astype(str).str.strip()
df_merged["GEOGRAPHY_CODE"] = df_merged["GEOGRAPHY_CODE"].astype(str).str.strip()

# --- Step 1: Filter df_merged to 2022 rows ---
df_merged_2022 = df_merged[df_merged["YEAR"] == 2022].copy()

# --- Step 2: Pivot df_grouped to wide format ---
df_metrics_wide = df_grouped.pivot_table(
    index="Area Code",
    columns="Metric",
    values="Metric_Value",
    aggfunc="first"  # or "mean" if duplicates exist
).reset_index()

# Flatten columns if pivot_table created a MultiIndex
df_metrics_wide.columns.name = None

# --- Step 3: Merge 2022 metrics into 2022 rows only ---
df_merged_2022_final = df_merged_2022.merge(
    df_metrics_wide,
    left_on="GEOGRAPHY_CODE",
    right_on="Area Code",
    how="left"
)

# Drop the extra Area Code column from pivot
df_merged_2022_final = df_merged_2022_final.drop(columns=["Area Code"])

# --- Step 4: Optional: combine with other years back ---
df_final = pd.concat([
    df_merged[df_merged["YEAR"] != 2022],
    df_merged_2022_final
], ignore_index=True)

print(df_final.info())
df_final.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49700 entries, 0 to 49699
Data columns (total 57 columns):
 #   Column                                                                                                                                               Non-Null Count  Dtype   
---  ------                                                                                                                                               --------------  -----   
 0   YEAR                                                                                                                                                 49700 non-null  category
 1   GEOGRAPHY_CODE                                                                                                                                       49700 non-null  object  
 2   GEOGRAPHY_NAME                                                                                                                                       49700 non-null  categ

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment,% living in an area that has a devolution deal with a directly elected mayor,"19+ further education and skills participation (per 100,000 population)","Age-standardised mortality rate for those aged under 75 (per 100,000 population)",...,Proportion of cancers diagnosed at stages 1 and 2 (%),Proportion of children living with obesity at Year 6 age (%),Proportion of children living with obesity at reception age (%),Proportion of the population aged 16 to 64 with NVQ3+ qualification (%),Proportion of the population aged 16-64 with NVQ3+ qualification (%),Total FDI international investment position abroad at end period (£ million),Total FDI international investment position in the UK at end period (£ million),Total UK public-funded gross regional capital and non-capital expenditure on research and development (£ million),Total value of UK exports (£ million),Upper 95% Confidence Interval
0,2016,E06000001,Hartlepool,Advanced manufacturing,Aerospace manufacturing,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016,E06000001,Hartlepool,Advanced manufacturing,Agritech,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016,E06000001,Hartlepool,Advanced manufacturing,Automotive manufacturing,0.0,300.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016,E06000001,Hartlepool,Advanced manufacturing,Batteries,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016,E06000001,Hartlepool,Creative Industries,Advertising and marketing,10.0,50.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
df_final.to_csv("IS8_Business_Employment_Demography.csv") #Industry data on macroregion level + Demographic infos
df_merged_msoa.to_csv("MSOA_Business_Employment.csv") #Industry data on microregion level (we do not receive data on that granularity)

#Remember the demographic data is from 2022 only. So, the rest is missing value. 